# Students Performance in Exams — Exploratory Data Analysis

> **Objective:** Explore the factors associated with students' academic performance and test whether completing a test-preparation course is associated with higher total scores.

This notebook covers **data inspection, data quality checks, feature engineering, exploratory analysis, visualization, outlier detection, and hypothesis testing**.

### Dataset
The analysis uses the **Students Performance in Exams** dataset containing 1,000 student records and 8 original variables.

### Key questions
1. What does the overall score distribution look like?
2. How does performance vary by gender and race/ethnicity group?
3. Is parental education associated with student performance?
4. Do students who completed the test-preparation course score higher?
5. Is the difference in total scores statistically significant?

## Analysis Roadmap

1. **Setup & Data Loading**
2. **Data Understanding & Quality Checks**
3. **Feature Engineering**
4. **Exploratory Data Analysis**
   - Test-preparation participation
   - Score distributions
   - Parental education
   - Gender
   - Race/ethnicity
5. **Outlier Analysis**
6. **Hypothesis Testing**
7. **Key Findings & Conclusion**

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind

import plotly.express as px

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [ ]:
data = pd.read_csv(
    "/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv"
)

data.head()

## 2. Data Understanding & Quality Checks

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
missing_values = data.isna().sum().sort_values(ascending=False)
missing_values

In [ ]:
duplicate_count = data.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

In [ ]:
data.nunique().sort_values()

### Initial observations

- The dataset contains **1,000 students** and **8 original variables**.
- The dataset has **no missing values**.
- The categorical variables contain a manageable number of groups, making them suitable for grouped analysis.
- The three score variables are numeric and range from 0 to 100.

## 3. Feature Engineering

In [ ]:
score_columns = ["math score", "reading score", "writing score"]

data["average score"] = data[score_columns].mean(axis=1)
data["total score"] = data[score_columns].sum(axis=1)

data[score_columns + ["average score", "total score"]].head()

In [ ]:
top_students = (
    data.sort_values("total score", ascending=False)
        .loc[:, [
            "gender",
            "race/ethnicity",
            "parental level of education",
            "test preparation course",
            *score_columns,
            "average score",
            "total score"
        ]]
)

top_students.head(10)

## 4. Exploratory Data Analysis

### 4.1 Test-Preparation Course Participation

In [ ]:
prep_counts = data["test preparation course"].value_counts()

prep_counts

In [ ]:
fig = px.bar(
    prep_counts.reset_index(),
    x="test preparation course",
    y="count",
    title="Test-Preparation Course Participation",
    labels={
        "test preparation course": "Course Status",
        "count": "Number of Students"
    },
    text="count"
)

fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

**Observation:** 358 students completed the preparation course, while 642 did not. Therefore, most students in this dataset did not complete the course.

### 4.2 Overall Score Distribution

In [ ]:
fig = px.histogram(
    data,
    x="total score",
    nbins=20,
    marginal="box",
    title="Distribution of Total Scores",
    labels={"total score": "Total Score"}
)

fig.update_layout(bargap=0.08)
fig.show()

In [ ]:
fig = px.box(
    data,
    y="average score",
    points="outliers",
    title="Distribution of Students' Average Scores",
    labels={"average score": "Average Score"}
)

fig.show()

### 4.3 Parental Level of Education

In [ ]:
education_avg = (
    data.groupby("parental level of education")["total score"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
)

education_avg

In [ ]:
fig = px.bar(
    education_avg,
    x="parental level of education",
    y="total score",
    title="Average Total Score by Parental Education",
    labels={
        "parental level of education": "Parental Education",
        "total score": "Average Total Score"
    },
    text="total score"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(xaxis_tickangle=-25)
fig.show()

**Observation:** Students whose parents hold a master's degree have the highest average total score in this dataset, while the high-school group has the lowest average.

### 4.4 Performance by Gender

In [ ]:
gender_scores = (
    data.groupby("gender")[score_columns]
        .mean()
        .reset_index()
)

gender_scores["average score"] = gender_scores[score_columns].mean(axis=1)

gender_scores

In [ ]:
fig = px.bar(
    gender_scores,
    x="gender",
    y=score_columns,
    barmode="group",
    title="Average Subject Scores by Gender",
    labels={
        "gender": "Gender",
        "value": "Average Score",
        "variable": "Subject"
    },
    text_auto=".2f"
)

fig.show()

In [ ]:
gender_overall = (
    data.groupby("gender")["average score"]
        .mean()
        .reset_index()
)

fig = px.bar(
    gender_overall,
    x="gender",
    y="average score",
    title="Overall Average Score by Gender",
    labels={"average score": "Average Score"},
    text="average score"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

**Observation:** Female students have higher average Reading and Writing scores, while male students have a higher average Math score. Overall, the female group has the higher average score.

### 4.5 Test Preparation vs. Total Score

In [ ]:
prep_scores = (
    data.groupby("test preparation course")["total score"]
        .mean()
        .reset_index()
)

prep_scores

In [ ]:
fig = px.bar(
    prep_scores,
    x="test preparation course",
    y="total score",
    title="Average Total Score by Test-Preparation Status",
    labels={
        "test preparation course": "Course Status",
        "total score": "Average Total Score"
    },
    text="total score"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.show()

**Observation:** Students who completed the test-preparation course have a higher average total score than students who did not.

### 4.6 Relationship Between Math and Reading Scores

In [ ]:
fig = px.scatter(
    data,
    x="math score",
    y="reading score",
    trendline="ols",
    title="Math Score vs. Reading Score",
    labels={
        "math score": "Math Score",
        "reading score": "Reading Score"
    },
    opacity=0.65
)

fig.show()

### 4.7 Performance by Race/Ethnicity Group

In [ ]:
race_scores = (
    data.groupby("race/ethnicity")[score_columns]
        .mean()
        .reset_index()
)

race_scores

In [ ]:
fig = px.bar(
    race_scores,
    x="race/ethnicity",
    y=score_columns,
    barmode="group",
    title="Average Subject Scores by Race/Ethnicity",
    labels={
        "race/ethnicity": "Race/Ethnicity Group",
        "value": "Average Score",
        "variable": "Subject"
    },
    text_auto=".2f"
)

fig.show()

**Observation:** Group E has the highest average performance across Math, Reading, and Writing, while Group A has the lowest averages among the five groups in this dataset.

## 5. Outlier Analysis

In [ ]:
Q1 = data["total score"].quantile(0.25)
Q3 = data["total score"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[
    (data["total score"] < lower_bound) |
    (data["total score"] > upper_bound)
]

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")
print(f"Number of outliers: {len(outliers)}")

In [ ]:
outliers[
    [
        "gender",
        "race/ethnicity",
        "lunch",
        "test preparation course",
        *score_columns,
        "average score",
        "total score"
    ]
].sort_values("total score")

**Interpretation:** The IQR method identifies unusually low total scores. These observations are not automatically errors; they may represent genuinely low-performing students and should therefore be investigated rather than removed without justification.

## 6. Hypothesis Testing — Does Test Preparation Matter?

### Research question
Do students who completed the test-preparation course have a different mean total score from students who did not?

**Null hypothesis (H₀):** The mean total score is the same for both groups.

**Alternative hypothesis (H₁):** The mean total score differs between the two groups.

We use an **independent two-sample Welch's t-test**, which does not assume equal population variances.

In [ ]:
completed = data.loc[
    data["test preparation course"] == "completed",
    "total score"
]

not_completed = data.loc[
    data["test preparation course"] == "none",
    "total score"
]

t_stat, p_value = ttest_ind(
    completed,
    not_completed,
    equal_var=False
)

print(f"Completed group mean: {completed.mean():.2f}")
print(f"Not-completed group mean: {not_completed.mean():.2f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4e}")

In [ ]:
alpha = 0.05

if p_value < alpha:
    print(
        "Decision: Reject H₀. "
        "There is statistically significant evidence of a difference in mean total scores."
    )
else:
    print(
        "Decision: Fail to reject H₀. "
        "There is not enough evidence of a difference in mean total scores."
    )

### Statistical conclusion

The p-value is far below the 0.05 significance level, so we **reject the null hypothesis**. There is strong statistical evidence that the mean total scores differ between students who completed the preparation course and those who did not.

> **Important:** Statistical significance shows that the groups differ; it does **not** by itself prove that completing the course caused the higher scores. Other factors may also contribute to the difference.

## 7. Key Findings

### Main insights

- **Data quality:** No missing values were found in the original dataset.
- **Overall performance:** Reading and Writing have higher average scores than Math.
- **Test preparation:** 358 students completed the course, compared with 642 who did not.
- **Course performance:** Students who completed the preparation course have a higher average total score.
- **Hypothesis test:** Welch's t-test indicates a statistically significant difference in total scores between the two groups.
- **Gender:** Female students perform better on average in Reading and Writing, while male students perform better in Math.
- **Parental education:** The master's-degree group has the highest average total score.
- **Race/ethnicity:** Group E has the highest average across the three subjects, while Group A has the lowest.
- **Outliers:** The IQR method identifies a small number of unusually low total scores; these should be investigated rather than automatically removed.

### Final takeaway

The analysis suggests that **test preparation, student demographics, and parental education are associated with differences in academic performance** in this dataset. The strongest statistical result explored here is the significant difference in total scores between students who completed the test-preparation course and those who did not.

## Notes & Limitations

- This is an **observational dataset**, so statistical association should not be interpreted as causation.
- The race/ethnicity and gender categories are treated as the labels provided by the dataset; the analysis does not make broader claims about demographic groups.
- Outliers are retained because extreme scores can represent real observations.
- The hypothesis test compares group means and does not control for other variables such as lunch type, parental education, or gender.

---

**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Plotly · SciPy